# Breast cancer data

Data [10x Xenium breast cancer data](https://www.10xgenomics.com/products/xenium-in-situ/preview-dataset-human-breast).

## Library imports

In [1]:
import warnings
warnings.filterwarnings('ignore')

import os
import requests

from spatialdata_io import xenium
from tqdm.auto import tqdm
from zipfile import ZipFile

In [ ]:
def download_data(url, destination_folder, boolzip):
    print(f'Downloading data from {url} to {destination_folder}...') 
    
    os.makedirs(destination_folder, exist_ok=True)
    file_name = os.path.join(destination_folder, url.split("/")[-1])

    response = requests.get(url, stream=True)
    total_size = int(response.headers.get('content-length', 0))
    block_size = 1024 * 1024  # 1 MB

    with open(file_name, 'wb') as file, tqdm(
        total=total_size,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
        desc=f"Downloading {os.path.basename(file_name)}",
        bar_format="{desc}: {percentage:3.0f}% ({n_fmt}/{total_fmt})"
    ) as pbar:
        for data in response.iter_content(block_size):
            file.write(data)
            pbar.update(len(data))

    if boolzip:
        print("Extracting...")
        with ZipFile(file_name, 'r') as zip_ref:
            zip_ref.extractall(destination_folder)

    print('...done')

THREADS = 8
OUTDIR = "./Xenium_FFPE_Human_Breast_Cancer_Rep1_outs/"

In [ ]:
print(f'Download data ...')
download_data('https://cf.10xgenomics.com/samples/xenium/1.0.1/Xenium_FFPE_Human_Breast_Cancer_Rep1/Xenium_FFPE_Human_Breast_Cancer_Rep1_outs.zip', f'{OUTDIR}/', True)
download_data('https://cdn.10xgenomics.com/raw/upload/v1695234604/Xenium%20Preview%20Data/Cell_Barcode_Type_Matrices.xlsx', f'{OUTDIR}/', False)

Download data ...


Extracting...
...done


...done


In [4]:
# Do the conversion then its quicker to read in data later.
# Might take about 1-2 min with 8 cores.
print(f'Convert Xenium bundle to SpatialData bundle ...')
sd_xenium_obj = xenium(
            f"{OUTDIR}/outs",
            n_jobs=THREADS,
            cells_as_shapes=True,
            nucleus_boundaries=True,
            transcripts=True,
            morphology_mip=True,
            morphology_focus=True,
)
print(sd_xenium_obj)

Convert Xenium bundle to SpatialData bundle ...
INFO     reading Xenium_FFPE_Human_Breast_Cancer_Rep1_outs/outs/cell_feature_matrix.h5                             
SpatialData object
├── Images
│     ├── 'morphology_focus': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
│     └── 'morphology_mip': DataTree[cyx] (1, 25778, 35416), (1, 12889, 17708), (1, 6444, 8854), (1, 3222, 4427), (1, 1611, 2213)
├── Labels
│     ├── 'cell_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
│     └── 'nucleus_labels': DataTree[yx] (25778, 35416), (12889, 17708), (6444, 8854), (3222, 4427), (1611, 2213)
├── Points
│     └── 'transcripts': DataFrame with shape: (<Delayed>, 8) (3D points)
├── Shapes
│     ├── 'cell_boundaries': GeoDataFrame shape: (167780, 1) (2D shapes)
│     ├── 'cell_circles': GeoDataFrame shape: (167780, 2) (2D shapes)
│     └── 'nucleus_boundaries': GeoDataFrame shape: (167780, 1) (2D s

In [5]:
# Might take about 5-6 min
print("[NOTE] Write data")
sd_xenium_obj.write(f"{OUTDIR}/spatialdata")
print("[FINISH]")

[NOTE] Write data
INFO     The Zarr backing store has been changed from None the new file path:                                      
         Xenium_FFPE_Human_Breast_Cancer_Rep1_outs/spatialdata                                                     
[FINISH]
